# IFEST DAC 2026 — V7A IndoBART Conditional Headline Likelihood Verifier

**Frozen champion:** V5 (V2 train-reference structural graph → V4 test-test canonical matching → V5 asymmetric positive rescue).  
**New hypothesis:** the dataset is close to an article → canonical-headline construction process. V7A learns that process directly with IndoBART-v2 and uses teacher-forced conditional likelihood as an orthogonal factuality signal.

### Non-negotiable contract
- V2/V4 structural decisions and V5 rescue decisions are immutable.
- New prospective component-safe validation: 80% **DESIGN** / 20% untouched **LOCK**, split with seed `260907`.
- DESIGN uses 3-fold connected-component CV; LOCK labels are not inspected until coverage is frozen.
- IndoBART-v2, source length 768, target length 48, 2 epochs, LR `2e-5`, real same-content counterfactual ranking only.
- No external dataset, no synthetic negatives, no title-conditioned retrieval, no generic threshold tuning.
- Risk is frozen at `0.5 × percentile(mean headline NLL) + 0.5 × percentile(top-2 token NLL)`.
- Deployment searches only the precommitted low-coverage grid and optimizes **Macro F1**, not a 90%-precision veto rule.
- Final intervention is only `1 → 0` on V5-positive pure-fallback rows.
- If DESIGN, LOCK, or final test-count gates fail, output falls back exactly to frozen V5.

The notebook is intentionally compact (<1 MB) and embeds only the minimum frozen numeric arrays needed from prior runs. No hidden test labels are embedded.

> **Runtime revision:** legacy `indobenchmark-toolkit` tokenizer dependency is replaced by an encode-equivalent direct SentencePiece wrapper to remain compatible with current Kaggle Transformers/Python runtimes. Experimental methodology is unchanged.


In [1]:
import os, sys, re, io, gc, json, math, time, base64, hashlib, random, zipfile, unicodedata, warnings, subprocess, importlib.util, tempfile, shutil
from pathlib import Path
from collections import defaultdict
from contextlib import nullcontext

# Minimal runtime dependencies.
# IMPORTANT: do not import indobenchmark-toolkit here. Its IndoNLGTokenizer still
# imports TensorFlow/JAX compatibility helpers removed by recent Kaggle
# Transformers releases. We use the checkpoint's SentencePiece model directly,
# which is token-ID equivalent for the encode-only path required by V7A.
requirements=[]
for mod,pkg in [
    ("transformers","transformers>=4.44,<6"),
    ("sentencepiece","sentencepiece>=0.2"),
    ("sklearn","scikit-learn"),
    ("huggingface_hub","huggingface_hub>=0.24"),
]:
    if importlib.util.find_spec(mod) is None: requirements.append(pkg)
if requirements:
    print("Installing missing dependencies:",requirements)
    subprocess.check_call([sys.executable,"-m","pip","install","-q",*requirements])

import numpy as np, pandas as pd
from tqdm.auto import tqdm
from IPython.display import display
from sklearn.metrics import f1_score
from sklearn.model_selection import StratifiedGroupKFold
import torch, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import AutoModelForSeq2SeqLM, get_linear_schedule_with_warmup
from huggingface_hub import hf_hub_download
import sentencepiece as spm
import transformers, sklearn

warnings.filterwarnings("ignore")
os.environ.setdefault("TOKENIZERS_PARALLELISM","false")
os.environ.setdefault("HF_HUB_DISABLE_TELEMETRY","1")
pd.set_option("display.max_colwidth",180)

# ---------------- V7A FROZEN CONFIG ----------------
SEED=2026
LOCK_SEED=260907
DESIGN_FOLDS=3
EXPECTED_TRAIN_ROWS=14397; EXPECTED_TEST_ROWS=3603
TRAIN_ID_SHA256="9abdabdd6cda9baac679b5cf004572e02326859fc2b13caa37af32c335e6c94a"
TEST_ID_SHA256="e4763272ee9a832ad2141f979a48701222a6651550ba2a88cf6c89357094064c"

# Frozen V2/V4/V5 constants — never retune in V7A.
V2_TITLE_GATE=0.9839511586057019; MIN_GRAPH_CONTENT_WORDS=20
RANK_V3_WEIGHT=0.45; RANK_V2_GLOBAL_WEIGHT=0.55
CONTENT_CANONICAL_MIN_SCORE=0.20; TITLE_CANONICAL_MIN_SCORE=0.90; TITLE_CANONICAL_MIN_MARGIN=0.02
V5_BLEND_MIN=0.67; V5_GLOBAL_ENTAIL_MIN=0.50
V2_META_THRESHOLD=0.5969828254552141  # frozen from the already-run V2 OOF experiment

# IndoBART conditional-likelihood verifier.
MODEL_NAME="indobenchmark/indobart-v2"
MODEL_REVISION="e5b7d7ca907c6ba95cd361270b568f7ee9ceb0fd"
LOCAL_MODEL_PATH=os.environ.get("V7A_MODEL_PATH","").strip() or None
SOURCE_MAX=768; TARGET_MAX=48
EPOCHS=2; LR=2.0e-5; WEIGHT_DECAY=0.01; WARMUP_RATIO=0.05
TRAIN_BATCH=4; PAIR_BATCH=2; GRAD_ACCUM=8; SCORE_BATCH=8
PAIR_LAMBDA=0.50; MAX_GRAD_NORM=1.0; USE_AMP=True; USE_GRADIENT_CHECKPOINTING=True

# Frozen risk / coverage policy.
RISK_MEAN_WEIGHT=0.50; RISK_TOP2_WEIGHT=0.50
COVERAGE_GRID=np.array([0.0,0.002,0.004,0.006,0.010,0.015,0.020,0.030],dtype=float)

# Prospective promotion gates.
MIN_DESIGN_GAIN=0.0015
MIN_POSITIVE_DESIGN_FOLDS=2
MIN_WORST_DESIGN_FOLD_GAIN=-0.0005
MIN_PAIR_RANK_ACC=0.90
MIN_LOCK_GAIN=0.0
MIN_LOCK_PRECISION=0.50
BOOTSTRAP_REPS=1000
MIN_BOOTSTRAP_POSITIVE=0.80
MIN_TEST_CHANGES=5; MAX_TEST_CHANGES=90

OUTPUT_DIR=Path("/kaggle/working/ifest_v7a_output" if Path("/kaggle/working").exists() else "/mnt/data/ifest_v7a_output")
if OUTPUT_DIR.exists(): shutil.rmtree(OUTPUT_DIR)
OUTPUT_DIR.mkdir(parents=True,exist_ok=True)
TMP_DIR=OUTPUT_DIR/"_tmp_checkpoints"; TMP_DIR.mkdir(exist_ok=True)


def seed_everything(seed=SEED):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)
    if hasattr(torch.backends,"cudnn"):
        torch.backends.cudnn.deterministic=True; torch.backends.cudnn.benchmark=False
seed_everything()

DEVICE=torch.device("cuda" if torch.cuda.is_available() else "cpu")
GPU_COUNT=torch.cuda.device_count(); AMP_ENABLED=bool(USE_AMP and DEVICE.type=="cuda")
if DEVICE.type!="cuda": raise RuntimeError("V7A training requires Kaggle GPU. Select the T4 x2 accelerator before running.")
print("GPU count:",GPU_COUNT)
for i in range(GPU_COUNT):
    p=torch.cuda.get_device_properties(i); print(i,torch.cuda.get_device_name(i),f"VRAM={p.total_memory/1024**3:.1f} GB")
print(f"PyTorch={torch.__version__} | Transformers={transformers.__version__} | sklearn={sklearn.__version__} | AMP={AMP_ENABLED}")

GPU count: 2
0 Tesla T4 VRAM=14.6 GB
1 Tesla T4 VRAM=14.6 GB
PyTorch=2.10.0+cu128 | Transformers=5.0.0 | sklearn=1.6.1 | AMP=True


In [2]:
# Minimal frozen arrays from already-run V2/V3/V5. No hidden target labels.
_FROZEN_B64="""..."""
blob=base64.b64decode(_FROZEN_B64.encode())
assert hashlib.sha256(blob).hexdigest()=="e1480451c5d635d180201ae1957c75f6ec0383c99be9f5fb930928c824c29073"
_npz=np.load(io.BytesIO(blob),allow_pickle=False)
oof_v2_meta=_npz["oof_v2_meta"].astype(np.float64)
oof_v2_global=_npz["oof_v2_global"].astype(np.float64)
oof_v2_meta_label=_npz["oof_v2_meta_label"].astype(np.int8)
oof_v2_rule_code=_npz["oof_v2_rule"].astype(np.int8)
oof_v3_blend=_npz["oof_v3_blend"].astype(np.float64)
test_v5_label=_npz["test_v5_label"].astype(np.int8)
test_v4_rule_code=_npz["test_v4_rule"].astype(np.int8)
test_v5_rescue=_npz["test_v5_rescue"].astype(bool)
del _npz, blob
assert len(oof_v2_meta)==EXPECTED_TRAIN_ROWS and len(test_v5_label)==EXPECTED_TEST_ROWS
print("Frozen minimal prior arrays loaded.")

Frozen minimal prior arrays loaded.


In [3]:
def find_competition_csv(filename,expected_cols,expected_rows):
    roots=[Path("/kaggle/input"),Path("/mnt/data/ifest_dac")]; candidates=[]
    for root in roots:
        if root.exists(): candidates.extend(root.rglob(filename))
    candidates=[p for p in candidates if "/kaggle/working/" not in str(p)]
    valid=[]
    for p in candidates:
        try:
            h=pd.read_csv(p,nrows=3)
            if list(h.columns)==expected_cols:
                d=pd.read_csv(p)
                if len(d)==expected_rows: valid.append((p,d))
        except Exception: pass
    if not valid: raise FileNotFoundError(f"Could not locate a valid {filename}. Attach the IFEST competition dataset.")
    valid.sort(key=lambda z:(len(z[0].parts),len(str(z[0]))))
    return valid[0]

train_path,train=find_competition_csv("train.csv",["id","title","content","label"],EXPECTED_TRAIN_ROWS)
test_path,test=find_competition_csv("test.csv",["id","title","content"],EXPECTED_TEST_ROWS)
sample_path,sample=find_competition_csv("sample_submission.csv",["id","label"],EXPECTED_TEST_ROWS)
assert train.label.isin([0,1]).all()
def id_sha(df): return hashlib.sha256("\n".join(df.id.astype(str)).encode()).hexdigest()
assert id_sha(train)==TRAIN_ID_SHA256 and id_sha(test)==TEST_ID_SHA256
assert sample.id.astype(str).tolist()==test.id.astype(str).tolist()
print("Dataset:",train_path.parent,"| shapes:",train.shape,test.shape)

Dataset: /kaggle/input/competitions/penyisihan-ifest-2026-dac | shapes: (14397, 4) (3603, 3)


## Frozen V5 reconstruction
The prior champion is reconstructed without training. Its OOF decisions remain the baseline for the new prospective DESIGN/LOCK experiment; V7A is not allowed to alter V2/V4 structural rows or V5 rescues.

In [4]:
TOKEN_RE=re.compile(r"[\w]+",flags=re.UNICODE)
SOURCE_SUFFIX_RE=re.compile(r"\s+kumparan\s+com\s*$",re.I)

def clean_ws(x): return re.sub(r"\s+"," ",unicodedata.normalize("NFKC",str(x)).replace("\xa0"," ")).strip()
def clean_title_for_model(x): return SOURCE_SUFFIX_RE.sub("",clean_ws(x)).strip()
def normalize_key(x): return " ".join(TOKEN_RE.findall(clean_ws(x).lower()))

for df in (train,test):
    df["title_model"]=df.title.map(clean_title_for_model)
    df["title_key"]=df.title_model.map(normalize_key)
    df["content_key"]=df.content.map(normalize_key)
    df["pair_key"]=df.title_key+"|||"+df.content_key
    df["content_words"]=df.content.map(lambda x:len(clean_ws(x).split()))

def build_graph_reference(ref):
    pair_to_label={}
    for k,g in ref.groupby("pair_key",sort=False):
        labs=set(map(int,g.label))
        if len(labs)==1: pair_to_label[k]=next(iter(labs))
    cpos=defaultdict(set); tpos=defaultdict(set)
    for r in ref.loc[ref.label==1].itertuples():
        cpos[r.content_key].add(r.title_key); tpos[r.title_key].add(r.content_key)
    return pair_to_label,cpos,tpos

def frozen_v2_graph(qry,ref,meta):
    pair,cpos,tpos=build_graph_reference(ref); rules=np.array(["semantic"]*len(qry),object); forced=np.full(len(qry),-1,np.int8)
    for i,r in enumerate(qry.itertuples()):
        if r.pair_key in pair:
            forced[i]=pair[r.pair_key]; rules[i]="exact_pair"; continue
        malformed=r.content_words<MIN_GRAPH_CONTENT_WORDS; pt=cpos.get(r.content_key,set())
        if not malformed and len(pt)==1 and r.title_key not in pt:
            forced[i]=0; rules[i]="unique_content_different_title"; continue
        pc=tpos.get(r.title_key,set())
        if not malformed and pc and r.content_key not in pc and meta[i]<V2_TITLE_GATE:
            forced[i]=0; rules[i]="gated_title_different_content"
    return rules,forced

# V2 semantic threshold is frozen from the prior experiment; it is not re-selected in V7A.
rank_oof=RANK_V3_WEIGHT*oof_v3_blend+RANK_V2_GLOBAL_WEIGHT*oof_v2_global

def base_v2(qry,ref,meta):
    lab=(np.asarray(meta)>=V2_META_THRESHOLD).astype(np.int8); rules,forced=frozen_v2_graph(qry,ref,np.asarray(meta))
    m=forced>=0; lab[m]=forced[m]; return lab,rules,forced

def apply_v4(qry,base,forced,rank,ref):
    q=qry.reset_index(drop=True); lab=np.asarray(base,np.int8).copy(); forced=np.asarray(forced,np.int8); rank=np.asarray(rank,float)
    _,cpos,tpos=build_graph_reference(ref); rule=np.array(["v2_fallback"]*len(q),object); lock=forced>=0; rule[lock]="v2_graph_locked"
    for ckey,g0 in q.loc[~lock].groupby("content_key",sort=False):
        if g0.content_words.iloc[0]<MIN_GRAPH_CONTENT_WORDS or g0.title_key.nunique()<2 or len(cpos.get(ckey,set()))>0: continue
        idx=g0.index.to_numpy(); cg=(g0.assign(_s=rank[idx]).groupby("title_key",as_index=False).agg(score=("_s","mean")).sort_values(["score","title_key"],ascending=[False,True]))
        if float(cg.score.iloc[0])<CONTENT_CANONICAL_MIN_SCORE: continue
        winner=cg.title_key.iloc[0]; lab[idx]=(g0.title_key.to_numpy()==winner).astype(np.int8); rule[idx]="test_content_canonical"; lock[idx]=True
    for tkey,g0 in q.loc[~lock].groupby("title_key",sort=False):
        if g0.content_key.nunique()<2 or len(tpos.get(tkey,set()))>0: continue
        idx=g0.index.to_numpy(); cg=(g0.assign(_s=rank[idx]).groupby("content_key",as_index=False).agg(score=("_s","mean")).sort_values(["score","content_key"],ascending=[False,True]))
        if len(cg)<2: continue
        top,sec=float(cg.score.iloc[0]),float(cg.score.iloc[1])
        if top<TITLE_CANONICAL_MIN_SCORE or top-sec<TITLE_CANONICAL_MIN_MARGIN: continue
        winner=cg.content_key.iloc[0]; lab[idx]=(g0.content_key.to_numpy()==winner).astype(np.int8); rule[idx]="test_title_canonical"; lock[idx]=True
    return lab,rule

def apply_v5(v4,rule,blend,glob):
    lab=np.asarray(v4,np.int8).copy(); rescue=(lab==0)&(np.asarray(rule)=="v2_fallback")&(blend>=V5_BLEND_MIN)&(glob>=V5_GLOBAL_ENTAIL_MIN)
    lab[rescue]=1; return lab,rescue

class UF:
    def __init__(self,n): self.p=list(range(n)); self.r=[0]*n
    def find(self,x):
        while self.p[x]!=x: self.p[x]=self.p[self.p[x]]; x=self.p[x]
        return x
    def union(self,a,b):
        a,b=self.find(a),self.find(b)
        if a==b:return
        if self.r[a]<self.r[b]:a,b=b,a
        self.p[b]=a
        if self.r[a]==self.r[b]:self.r[a]+=1

def component_groups(df):
    u=UF(len(df))
    for col in ["title_key","content_key"]:
        seen={}
        for i,k in enumerate(df[col]):
            if k in seen:u.union(i,seen[k])
            else:seen[k]=i
    roots=[u.find(i) for i in range(len(df))]; mp={r:j for j,r in enumerate(dict.fromkeys(roots))}
    return np.array([mp[r] for r in roots],np.int32)

groups=component_groups(train)
old_cv=StratifiedGroupKFold(n_splits=3,shuffle=True,random_state=SEED)
old_fold=np.full(len(train),-1,np.int8); oof_v5=np.full(len(train),-1,np.int8); oof_v4_rule=np.array([""]*len(train),object); oof_v5_rescue=np.zeros(len(train),bool)
for f,(tr,va) in enumerate(old_cv.split(train,train.label,groups)):
    old_fold[va]=f; ref=train.iloc[tr].reset_index(drop=True); qry=train.iloc[va].reset_index(drop=True)
    b,_,fr=base_v2(qry,ref,oof_v2_meta[va]); v4,r4=apply_v4(qry,b,fr,rank_oof[va],ref); v5,res=apply_v5(v4,r4,oof_v3_blend[va],oof_v2_global[va])
    oof_v5[va]=v5; oof_v4_rule[va]=r4; oof_v5_rescue[va]=res
assert (oof_v5>=0).all()
y=train.label.to_numpy(np.int8)
test_v4_rule=np.array(["v2_fallback","v2_graph_locked","test_content_canonical","test_title_canonical"],object)[np.clip(test_v4_rule_code,0,3)]
train_candidate=(oof_v5==1)&(oof_v4_rule=="v2_fallback")&(~oof_v5_rescue)
test_candidate=(test_v5_label==1)&(test_v4_rule=="v2_fallback")&(~test_v5_rescue)
print("Frozen V5 OOF Macro F1:",f1_score(y,oof_v5,average="macro",zero_division=0))
print("Frozen test V5 distribution:",pd.Series(test_v5_label).value_counts().sort_index().to_dict(),"| eligible:",int(test_candidate.sum()))

Frozen V5 OOF Macro F1: 0.8946909330297752
Frozen test V5 distribution: {0: 379, 1: 3224} | eligible: 2255


## Prospective DESIGN / LOCK split
The LOCK partition is created now but its labels are not accessed by the V7A selection logic. Coverage is selected using DESIGN OOF only; LOCK is opened once, after the risk function and coverage are frozen.

In [5]:
# 80/20 component-safe prospective split. First fold of a 5-fold SGKF is the untouched LOCK.
outer=StratifiedGroupKFold(n_splits=5,shuffle=True,random_state=LOCK_SEED)
design_idx,lock_idx=next(outer.split(train,train.label,groups))
design_idx=np.asarray(design_idx); lock_idx=np.asarray(lock_idx)
assert set(groups[design_idx]).isdisjoint(set(groups[lock_idx]))
assert 0.15 < len(lock_idx)/len(train) < 0.25

# DESIGN-only inner component folds.
inner=StratifiedGroupKFold(n_splits=DESIGN_FOLDS,shuffle=True,random_state=LOCK_SEED)
design_fold=np.full(len(train),-1,np.int8); design_splits=[]
for f,(a,b) in enumerate(inner.split(train.iloc[design_idx],train.label.iloc[design_idx],groups[design_idx])):
    tr=design_idx[np.asarray(a)]; va=design_idx[np.asarray(b)]
    assert set(groups[tr]).isdisjoint(set(groups[va])); design_fold[va]=f; design_splits.append((tr,va))
assert (design_fold[design_idx]>=0).all() and (design_fold[lock_idx]==-1).all()

split_table=pd.DataFrame({"id":train.id,"component":groups,"partition":np.where(np.isin(np.arange(len(train)),lock_idx),"LOCK","DESIGN"),"design_fold":design_fold})
print("DESIGN/LOCK sizes:",len(design_idx),len(lock_idx),"| component overlap:",len(set(groups[design_idx])&set(groups[lock_idx])))
print("DESIGN label prevalence:",float(train.label.iloc[design_idx].mean()))
print("LOCK remains sealed until the post-selection cell.")

def positive_examples(indices):
    d=train.iloc[np.asarray(indices)]
    d=d[d.label==1].drop_duplicates(["content_key","title_key"])
    return [(str(r.content),str(r.title_model)) for r in d.itertuples()]

def contrastive_examples(indices):
    d=train.iloc[np.asarray(indices)]; out=[]
    for ck,g in d.groupby("content_key",sort=False):
        pos=g[g.label==1]; neg=g[g.label==0]
        pkeys=pos.title_key.drop_duplicates().tolist()
        if len(pkeys)!=1 or len(neg)==0: continue
        prow=pos[pos.title_key==pkeys[0]].iloc[0]
        for _,ng in neg.groupby("title_key",sort=False):
            nrow=ng.iloc[0]
            out.append((str(prow.content),str(prow.title_model),str(nrow.title_model),str(ck)))
    return out

design_pos=positive_examples(design_idx); design_pairs=contrastive_examples(design_idx)
print("DESIGN distinct positive mappings:",len(design_pos),"| DESIGN real same-content contrastive pairs:",len(design_pairs))
del design_pos,design_pairs

DESIGN/LOCK sizes: 11548 2849 | component overlap: 0
DESIGN label prevalence: 0.899030135088327
LOCK remains sealed until the post-selection cell.
DESIGN distinct positive mappings: 8925 | DESIGN real same-content contrastive pairs: 733


## IndoBART-v2 training and teacher-forced scoring
The official IndoNLG tokenizer format is used: encoder inputs end with the Indonesian language token, and labels end with the same language token so MBART shifts it to the decoder start. Pair ranking uses length-normalized headline-token NLL.

In [6]:
def resolve_model_source():
    if LOCAL_MODEL_PATH: return LOCAL_MODEL_PATH
    root=Path("/kaggle/input")
    if root.exists():
        for cfg in root.rglob("config.json"):
            p=cfg.parent
            if (p/"sentencepiece.bpe.model").exists() and ((p/"pytorch_model.bin").exists() or (p/"model.safetensors").exists()):
                try:
                    z=json.loads(cfg.read_text())
                    if z.get("model_type") in {"mbart","bart"} and int(z.get("vocab_size",0))==40004:
                        print("Using local IndoBART checkpoint:",p); return str(p)
                except Exception: pass
    return MODEL_NAME

def hf_kwargs(src): return {} if src!=MODEL_NAME else {"revision":MODEL_REVISION}

class IndoBARTSentencePieceTokenizer:
    """Minimal encode-only tokenizer matching IndoNLGTokenizer token IDs.

    IndoBART uses a 40,000-piece SentencePiece vocabulary plus four Fairseq
    special IDs [javanese]=40000, [sundanese]=40001, [indonesian]=40002,
    <mask>=40003. V7A only needs encode() and those IDs; bypassing the legacy
    toolkit avoids its incompatibility with modern Transformers on Kaggle.
    """
    def __init__(self,vocab_file):
        self.vocab_file=str(vocab_file)
        self.sp_model=spm.SentencePieceProcessor(model_file=self.vocab_file)
        if int(self.sp_model.get_piece_size())!=40000:
            raise RuntimeError(f"Unexpected IndoBART SentencePiece size: {self.sp_model.get_piece_size()} (expected 40000)")
        self.bos_token_id=0; self.pad_token_id=1; self.eos_token_id=2; self.unk_token_id=3
        self.javanese_token_id=40000; self.sundanese_token_id=40001
        self.indonesian_token_id=40002; self.mask_token_id=40003
    def encode(self,text,add_special_tokens=False):
        ids=list(self.sp_model.encode(str(text).lower(),out_type=int))
        if add_special_tokens: return [self.bos_token_id]+ids+[self.eos_token_id]
        return ids
    def __len__(self): return 40004

def resolve_tokenizer_file(src):
    p=Path(str(src))
    if p.exists():
        vocab=p/"sentencepiece.bpe.model"
        if not vocab.exists():
            raise FileNotFoundError(f"sentencepiece.bpe.model not found under local checkpoint: {p}")
        return vocab
    return Path(hf_hub_download(repo_id=src,filename="sentencepiece.bpe.model",revision=MODEL_REVISION))

def load_tokenizer_model():
    src=resolve_model_source(); kw=hf_kwargs(src)
    try:
        tok=IndoBARTSentencePieceTokenizer(resolve_tokenizer_file(src))
        model=AutoModelForSeq2SeqLM.from_pretrained(src,torch_dtype=torch.float32,**kw)
    except Exception as e:
        raise RuntimeError("Failed to load IndoBART-v2. Enable Kaggle internet or attach the indobenchmark/indobart-v2 checkpoint and set V7A_MODEL_PATH.") from e
    model.float(); model.config.use_cache=False
    if USE_GRADIENT_CHECKPOINTING and hasattr(model,"gradient_checkpointing_enable"): model.gradient_checkpointing_enable()
    assert int(model.config.vocab_size)==len(tok)==40004
    return tok,model,src

def pad_2d(rows,pad_value,dtype=torch.long):
    m=max(len(x) for x in rows); out=torch.full((len(rows),m),pad_value,dtype=dtype)
    for i,x in enumerate(rows): out[i,:len(x)]=torch.as_tensor(x,dtype=dtype)
    return out

def encode_batch(tok,sources,targets):
    # SentencePiece wrapper lowercases internally, matching IndoBART pretraining.
    bos,eos,pad,lang=tok.bos_token_id,tok.eos_token_id,tok.pad_token_id,tok.indonesian_token_id
    src_ids=[]; labels=[]
    for s,t in zip(sources,targets):
        si=tok.encode(clean_ws(s),add_special_tokens=False)[:SOURCE_MAX-3]
        ti=tok.encode(clean_title_for_model(t),add_special_tokens=False)[:TARGET_MAX-3]
        src_ids.append([bos]+si+[eos,lang]); labels.append([bos]+ti+[eos,lang])
    input_ids=pad_2d(src_ids,pad); attention=(input_ids!=pad).long(); lab=pad_2d(labels,-100)
    return {"input_ids":input_ids,"attention_mask":attention,"labels":lab}

def title_token_mask(labels,tok):
    m=labels.ne(-100)
    for sid in [tok.bos_token_id,tok.eos_token_id,tok.indonesian_token_id,tok.pad_token_id]: m &= labels.ne(int(sid))
    return m

def per_sequence_nll(logits,labels,tok):
    # Loss math is explicitly FP32 and excludes special tokens from likelihood diagnostics/ranking.
    B,L,V=logits.shape
    z=F.cross_entropy(logits.float().reshape(-1,V),labels.reshape(-1),ignore_index=-100,reduction="none").reshape(B,L)
    mask=title_token_mask(labels,tok); denom=mask.sum(1).clamp_min(1)
    mean=(z*mask).sum(1)/denom
    top2=[]
    for i in range(B):
        vals=z[i][mask[i]]
        if vals.numel()==0: top2.append(torch.tensor(0.0,device=z.device))
        else: top2.append(torch.topk(vals,k=min(2,vals.numel())).values.mean())
    return mean,torch.stack(top2)

class ListDataset(Dataset):
    def __init__(self,rows): self.rows=list(rows)
    def __len__(self): return len(self.rows)
    def __getitem__(self,i): return self.rows[i]

def list_collate(x): return x

def amp_ctx(): return torch.autocast("cuda",dtype=torch.float16) if AMP_ENABLED else nullcontext()
def make_scaler():
    try:return torch.amp.GradScaler("cuda",enabled=AMP_ENABLED)
    except Exception:return torch.cuda.amp.GradScaler(enabled=AMP_ENABLED)
def unwrap(m): return m.module if isinstance(m,torch.nn.DataParallel) else m

def model_to_runtime(model):
    model.to(DEVICE)
    if GPU_COUNT>=2: model=torch.nn.DataParallel(model,device_ids=list(range(GPU_COUNT)))
    return model

def make_loader(rows,batch,shuffle,seed):
    gen=torch.Generator(); gen.manual_seed(seed)
    return DataLoader(ListDataset(rows),batch_size=batch,shuffle=shuffle,collate_fn=list_collate,num_workers=0,pin_memory=True,generator=gen if shuffle else None)

def cycle_loader(loader):
    while True:
        for b in loader: yield b

def train_indobart(train_indices,seed,tag):
    seed_everything(seed)
    pos=positive_examples(train_indices); pairs=contrastive_examples(train_indices)
    if not pos or not pairs: raise RuntimeError(f"Empty training population for {tag}: positives={len(pos)}, pairs={len(pairs)}")
    tok,base,src=load_tokenizer_model(); model=model_to_runtime(base)
    pos_loader=make_loader(pos,TRAIN_BATCH,True,seed); pair_loader=make_loader(pairs,PAIR_BATCH,True,seed+991); pair_iter=cycle_loader(pair_loader)
    opt=torch.optim.AdamW(model.parameters(),lr=LR,weight_decay=WEIGHT_DECAY)
    updates_per_epoch=math.ceil(len(pos_loader)/GRAD_ACCUM); total_updates=updates_per_epoch*EPOCHS
    sched=get_linear_schedule_with_warmup(opt,int(total_updates*WARMUP_RATIO),total_updates)
    scaler=make_scaler(); hist=[]; opt.zero_grad(set_to_none=True); global_step=0
    for ep in range(EPOCHS):
        model.train(); sum_ce=sum_rank=sum_total=0.0; nstep=0
        for step,batch in enumerate(tqdm(pos_loader,desc=f"{tag} epoch {ep+1}/{EPOCHS}",leave=False)):
            srcs=[x[0] for x in batch]; tgts=[x[1] for x in batch]
            pb=encode_batch(tok,srcs,tgts); pb={k:v.to(DEVICE,non_blocking=True) for k,v in pb.items()}
            with amp_ctx(): out=model(**pb); ce=out.loss.float().mean()
            pair_batch=next(pair_iter); c=[x[0] for x in pair_batch]; pt=[x[1] for x in pair_batch]; nt=[x[2] for x in pair_batch]
            qb=encode_batch(tok,c+c,pt+nt); qb={k:v.to(DEVICE,non_blocking=True) for k,v in qb.items()}
            with amp_ctx(): qout=model(**qb)
            qmean,_=per_sequence_nll(qout.logits,qb["labels"],tok); bsz=len(pair_batch)
            rank=F.softplus(qmean[:bsz]-qmean[bsz:]).mean()
            loss=ce+PAIR_LAMBDA*rank
            scaler.scale(loss/GRAD_ACCUM).backward(); sum_ce+=float(ce.detach()); sum_rank+=float(rank.detach()); sum_total+=float(loss.detach()); nstep+=1
            do_update=((step+1)%GRAD_ACCUM==0) or (step+1==len(pos_loader))
            if do_update:
                scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(model.parameters(),MAX_GRAD_NORM)
                scaler.step(opt); scaler.update(); sched.step(); opt.zero_grad(set_to_none=True); global_step+=1
            del out,qout,pb,qb,loss,ce,rank
        hist.append(dict(tag=tag,seed=seed,epoch=ep+1,positive_examples=len(pos),contrastive_pairs=len(pairs),steps=nstep,updates=global_step,ce_loss=sum_ce/max(1,nstep),rank_loss=sum_rank/max(1,nstep),total_loss=sum_total/max(1,nstep),lr=float(sched.get_last_lr()[0])))
    return tok,model,src,hist

@torch.no_grad()
def score_examples(model,tok,examples,batch_size=SCORE_BATCH,desc="score"):
    # examples: list[(content,title)]
    if not examples:return np.array([],float),np.array([],float)
    loader=make_loader(examples,batch_size,False,SEED); means=[]; tops=[]; model.eval()
    for batch in tqdm(loader,desc=desc,leave=False):
        b=encode_batch(tok,[x[0] for x in batch],[x[1] for x in batch]); b={k:v.to(DEVICE,non_blocking=True) for k,v in b.items()}
        with amp_ctx(): out=model(**b)
        m,t=per_sequence_nll(out.logits,b["labels"],tok); means.extend(m.detach().cpu().numpy()); tops.extend(t.detach().cpu().numpy())
        del out,b,m,t
    return np.asarray(means,float),np.asarray(tops,float)

def save_checkpoint(model,path):
    core=unwrap(model); core.to("cpu"); state={k:v.detach().contiguous() for k,v in core.state_dict().items()}; torch.save(state,path); del state
    return str(path)

def restore_checkpoint(path):
    tok,model,src=load_tokenizer_model(); 
    try: state=torch.load(path,map_location="cpu",weights_only=True)
    except TypeError: state=torch.load(path,map_location="cpu")
    model.load_state_dict(state,strict=True); del state
    return tok,model_to_runtime(model),src

def dispose_model(model=None):
    if model is not None:
        try: unwrap(model).to("cpu")
        except Exception: pass
        del model
    gc.collect(); torch.cuda.empty_cache()

## DESIGN-only 3-fold CV
Each DESIGN validation row is scored by a model that never saw its connected component. Strong same-content counterfactual pairs are evaluated in the same held-out fold. Fold checkpoints are stored temporarily for the one-time LOCK evaluation.

In [7]:
design_mean=np.full(len(train),np.nan,float); design_top2=np.full(len(train),np.nan,float)
training_history=[]; pair_rows=[]; fold_ckpts=[]; model_source=""

for f,(tr_idx,va_idx) in enumerate(design_splits):
    print(f"\n=== DESIGN fold {f} | train={len(tr_idx)} valid={len(va_idx)} ===")
    tok,model,src,hist=train_indobart(tr_idx,SEED+f,f"design_fold{f}"); model_source=src; training_history.extend(hist)
    # V5-positive pure-fallback candidates only.
    cand_idx=va_idx[train_candidate[va_idx]]
    ex=[(str(train.content.iloc[i]),str(train.title_model.iloc[i])) for i in cand_idx]
    m,t=score_examples(model,tok,ex,desc=f"fold{f} candidate OOF")
    design_mean[cand_idx]=m; design_top2[cand_idx]=t
    # Held-out real same-content pair ranking.
    vp=contrastive_examples(va_idx)
    if vp:
        pos=[(a,b) for a,b,_,_ in vp]; neg=[(a,c) for a,_,c,_ in vp]
        pm,_=score_examples(model,tok,pos,desc=f"fold{f} pair positive"); nm,_=score_examples(model,tok,neg,desc=f"fold{f} pair negative")
        for j,(content,pt,nt,ck) in enumerate(vp):
            pair_rows.append(dict(fold=f,content_key=ck,pos_title=pt,neg_title=nt,pos_mean_nll=float(pm[j]),neg_mean_nll=float(nm[j]),correct=int(pm[j]<nm[j]),margin=float(nm[j]-pm[j])))
    ck=TMP_DIR/f"design_fold{f}.pt"; fold_ckpts.append(save_checkpoint(model,ck)); dispose_model(model); del tok
    print("Saved temporary fold checkpoint:",ck)

training_history=pd.DataFrame(training_history); pair_metrics=pd.DataFrame(pair_rows)
assert np.isfinite(design_mean[design_idx[train_candidate[design_idx]]]).all()
pair_rank_acc=float(pair_metrics.correct.mean()) if len(pair_metrics) else 0.0
print("Held-out counterfactual pair ranking accuracy:",pair_rank_acc,"| pairs:",len(pair_metrics))


=== DESIGN fold 0 | train=7685 valid=3863 ===


sentencepiece.bpe.model:   0%|          | 0.00/932k [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


pytorch_model.bin:   0%|          | 0.00/526M [00:00<?, ?B/s]

...

design_fold2 epoch 2/2:   0%|          | 0/1493 [00:00<?, ?it/s]

fold2 candidate OOF:   0%|          | 0/390 [00:00<?, ?it/s]

fold2 pair positive:   0%|          | 0/29 [00:00<?, ?it/s]

fold2 pair negative:   0%|          | 0/29 [00:00<?, ?it/s]

Saved temporary fold checkpoint: /kaggle/working/ifest_v7a_output/_tmp_checkpoints/design_fold2.pt
Held-out counterfactual pair ranking accuracy: 0.9154160982264665 | pairs: 733


## DESIGN coverage selection — Macro-F1 aligned
The risk formula and coverage grid are frozen. Coverage is chosen only from DESIGN OOF, with ties resolved toward the smaller intervention. LOCK labels remain unused in this cell.

In [8]:
def macro_f1(yt,yp): return f1_score(yt,yp,average="macro",labels=[0,1],zero_division=0)

def percentile_risk(mean_nll,top2_nll):
    a=pd.Series(mean_nll).rank(method="average",pct=True).to_numpy(float)
    b=pd.Series(top2_nll).rank(method="average",pct=True).to_numpy(float)
    return RISK_MEAN_WEIGHT*a+RISK_TOP2_WEIGHT*b

def select_top(indices,risks,fraction,ids):
    indices=np.asarray(indices,int); risks=np.asarray(risks,float)
    if fraction<=0 or len(indices)==0:return np.array([],int)
    n=min(len(indices),max(1,int(math.ceil(float(fraction)*len(indices)))))
    order=sorted(range(len(indices)),key=lambda j:(-float(risks[j]),str(ids[indices[j]])))
    return indices[np.asarray(order[:n],int)]

# Risk is percentile-normalized independently inside each DESIGN fold, matching future LOCK/test usage.
design_risk=np.full(len(train),np.nan,float)
for f,(_,va_idx) in enumerate(design_splits):
    ci=va_idx[train_candidate[va_idx]]
    design_risk[ci]=percentile_risk(design_mean[ci],design_top2[ci])

coverage_rows=[]
for frac in COVERAGE_GRID:
    pred=oof_v5.copy(); selected=[]
    for f,(_,va_idx) in enumerate(design_splits):
        ci=va_idx[train_candidate[va_idx]]; sel=select_top(ci,design_risk[ci],frac,train.id.to_numpy())
        pred[sel]=0; selected.extend(sel.tolist())
    base_f1=macro_f1(y[design_idx],oof_v5[design_idx])
    new_f1=macro_f1(y[design_idx],pred[design_idx])
    sel=np.asarray(selected,int); prec=float((y[sel]==0).mean()) if len(sel) else np.nan
    coverage_rows.append(dict(coverage=float(frac),selected=len(sel),base_macro_f1=base_f1,v7a_macro_f1=new_f1,gain=new_f1-base_f1,veto_precision=prec))
coverage_metrics=pd.DataFrame(coverage_rows).sort_values(["gain","coverage"],ascending=[False,True]).reset_index(drop=True)
SELECTED_COVERAGE=float(coverage_metrics.iloc[0].coverage)

# Freeze the chosen coverage now; no LOCK information has been accessed.
design_pred=oof_v5.copy(); design_fold_rows=[]
for f,(_,va_idx) in enumerate(design_splits):
    ci=va_idx[train_candidate[va_idx]]; sel=select_top(ci,design_risk[ci],SELECTED_COVERAGE,train.id.to_numpy()); design_pred[sel]=0
    b=macro_f1(y[va_idx],oof_v5[va_idx]); n=macro_f1(y[va_idx],design_pred[va_idx])
    design_fold_rows.append(dict(fold=f,rows=len(va_idx),candidate_rows=len(ci),selected=len(sel),base_macro_f1=b,v7a_macro_f1=n,gain=n-b,veto_precision=float((y[sel]==0).mean()) if len(sel) else np.nan))
design_fold_metrics=pd.DataFrame(design_fold_rows)
design_base=macro_f1(y[design_idx],oof_v5[design_idx]); design_new=macro_f1(y[design_idx],design_pred[design_idx])
design_gain=design_new-design_base
positive_folds=int((design_fold_metrics.gain>0).sum()); worst_fold=float(design_fold_metrics.gain.min())
DESIGN_OK=bool(SELECTED_COVERAGE>0 and design_gain>=MIN_DESIGN_GAIN and positive_folds>=MIN_POSITIVE_DESIGN_FOLDS and worst_fold>MIN_WORST_DESIGN_FOLD_GAIN and pair_rank_acc>=MIN_PAIR_RANK_ACC)
print("Selected coverage:",SELECTED_COVERAGE)
print(f"DESIGN Macro F1: {design_base:.6f} -> {design_new:.6f} | gain={design_gain:+.6f}")
print("Positive folds:",positive_folds,"| worst fold:",worst_fold,"| pair ranking:",pair_rank_acc,"| DESIGN_OK:",DESIGN_OK)
display(coverage_metrics.sort_values("coverage")); display(design_fold_metrics)

Selected coverage: 0.0
DESIGN Macro F1: 0.893167 -> 0.893167 | gain=+0.000000
Positive folds: 0 | worst fold: 0.0 | pair ranking: 0.9154160982264665 | DESIGN_OK: False


,coverage,selected,base_macro_f1,v7a_macro_f1,gain,veto_precision
0,0.000,0,0.893167,0.893167,0.000000,NaN
1,0.002,21,0.893167,0.890150,-0.003017,0.095238
2,0.004,39,0.893167,0.887281,-0.005886,0.076923
3,0.006,57,0.893167,0.885329,-0.007838,0.105263
4,0.010,95,0.893167,0.878957,-0.014210,0.073684
5,0.015,141,0.893167,0.872219,-0.020948,0.070922
6,0.020,189,0.893167,0.864915,-0.028252,0.063492
7,0.030,281,0.893167,0.851160,-0.042007,0.053381


,fold,rows,candidate_rows,selected,base_macro_f1,v7a_macro_f1,gain,veto_precision
0,0,3863,3060,0,0.887812,0.887812,0.0,NaN
1,1,3869,3163,0,0.898453,0.898453,0.0,NaN
2,2,3816,3119,0,0.893641,0.893641,0.0,NaN


## One-time untouched LOCK evaluation
LOCK is opened only if DESIGN passes. The frozen DESIGN coverage is applied without adjustment. The three DESIGN fold models score LOCK; raw likelihood signals are averaged before percentile normalization. No threshold or coverage is changed after seeing LOCK.

In [9]:
lock_metrics=pd.DataFrame(); lock_scores=pd.DataFrame(); bootstrap_df=pd.DataFrame(); LOCK_OPENED=False; LOCK_OK=False

if DESIGN_OK:
    LOCK_OPENED=True
    li=lock_idx[train_candidate[lock_idx]]
    means=[]; tops=[]
    lex=[(str(train.content.iloc[i]),str(train.title_model.iloc[i])) for i in li]
    for f,ck in enumerate(fold_ckpts):
        tok,model,_=restore_checkpoint(ck)
        m,t=score_examples(model,tok,lex,desc=f"LOCK fold-model {f}"); means.append(m); tops.append(t)
        dispose_model(model); del tok
    lm=np.mean(np.stack(means),axis=0); lt=np.mean(np.stack(tops),axis=0); lrisk=percentile_risk(lm,lt)
    lock_sel=select_top(li,lrisk,SELECTED_COVERAGE,train.id.to_numpy())
    # LOCK labels are accessed for the first time here, after coverage is frozen.
    lock_y=y[lock_idx].copy(); base=oof_v5[lock_idx].copy(); new=base.copy()
    pos={g:i for i,g in enumerate(lock_idx)}
    for g in lock_sel:new[pos[int(g)]]=0
    base_f1=macro_f1(lock_y,base); new_f1=macro_f1(lock_y,new); gain=new_f1-base_f1
    precision=float((y[lock_sel]==0).mean()) if len(lock_sel) else 0.0
    # Component bootstrap on untouched LOCK.
    rng=np.random.default_rng(LOCK_SEED+77); ug=np.unique(groups[lock_idx]); deltas=[]
    group_rows={g:np.where(groups[lock_idx]==g)[0] for g in ug}
    for b in range(BOOTSTRAP_REPS):
        sampled=rng.choice(ug,size=len(ug),replace=True); rows=np.concatenate([group_rows[g] for g in sampled])
        db=macro_f1(lock_y[rows],base[rows]); dn=macro_f1(lock_y[rows],new[rows]); deltas.append(dn-db)
    boot_pos=float((np.asarray(deltas)>0).mean())
    LOCK_OK=bool(gain>MIN_LOCK_GAIN and precision>=MIN_LOCK_PRECISION and boot_pos>=MIN_BOOTSTRAP_POSITIVE)
    lock_metrics=pd.DataFrame([dict(rows=len(lock_idx),candidate_rows=len(li),selected=len(lock_sel),coverage=SELECTED_COVERAGE,base_macro_f1=base_f1,v7a_macro_f1=new_f1,gain=gain,veto_precision=precision,bootstrap_positive_fraction=boot_pos,lock_ok=LOCK_OK)])
    lock_scores=pd.DataFrame({"id":train.id.iloc[li].to_numpy(),"mean_nll":lm,"top2_nll":lt,"risk":lrisk,"selected":np.isin(li,lock_sel).astype(int)})
    bootstrap_df=pd.DataFrame({"replicate":np.arange(BOOTSTRAP_REPS),"delta_macro_f1":deltas})
    print("LOCK opened exactly once."); display(lock_metrics)
else:
    print("DESIGN gate failed. LOCK remains sealed and V7A will fail-closed to V5.")

V7A_VALIDATED=bool(DESIGN_OK and LOCK_OPENED and LOCK_OK)
print("V7A prospective validation accepted:",V7A_VALIDATED)

DESIGN gate failed. LOCK remains sealed and V7A will fail-closed to V5.
V7A prospective validation accepted: False


## Final two-seed full-train fit and test intervention
Only a prospectively validated V7A is allowed to train final export models. Both seeds use all official training labels. Test coverage remains frozen from DESIGN; no test-specific threshold tuning occurs.

In [10]:
final_v7a=test_v5_label.copy(); test_mean=np.full(len(test),np.nan); test_top2=np.full(len(test),np.nan); test_risk=np.full(len(test),np.nan); proposed=np.zeros(len(test),bool)
final_seed_history=[]; FINAL_SEEDS=[SEED,SEED+1]; TEST_COUNT_OK=False

if V7A_VALIDATED:
    ti=np.where(test_candidate)[0]; tex=[(str(test.content.iloc[i]),str(test.title_model.iloc[i])) for i in ti]
    seed_m=[]; seed_t=[]
    for s in FINAL_SEEDS:
        tok,model,src,hist=train_indobart(np.arange(len(train)),s,f"final_seed{s}"); final_seed_history.extend(hist)
        m,t=score_examples(model,tok,tex,desc=f"test seed {s}"); seed_m.append(m); seed_t.append(t)
        dispose_model(model); del tok
    test_mean[ti]=np.mean(np.stack(seed_m),axis=0); test_top2[ti]=np.mean(np.stack(seed_t),axis=0)
    test_risk[ti]=percentile_risk(test_mean[ti],test_top2[ti])
    sel=select_top(ti,test_risk[ti],SELECTED_COVERAGE,test.id.to_numpy()); proposed[sel]=True
    TEST_COUNT_OK=bool(MIN_TEST_CHANGES<=len(sel)<=MAX_TEST_CHANGES)
    if TEST_COUNT_OK: final_v7a[sel]=0
    else: proposed[:]=False; final_v7a=test_v5_label.copy()
    print("Frozen coverage:",SELECTED_COVERAGE,"| proposed test changes:",len(sel),"| test-count gate:",TEST_COUNT_OK)
else:
    print("Prospective validation failed; final model training skipped and predictions remain frozen V5.")

# Locked champion decisions must remain immutable.
assert not np.any(proposed & (test_v4_rule!="v2_fallback"))
assert not np.any(proposed & test_v5_rescue)
V7A_ACCEPTED=bool(V7A_VALIDATED and TEST_COUNT_OK)
print("V7A final accepted:",V7A_ACCEPTED,"| changed rows:",int((final_v7a!=test_v5_label).sum()))

Prospective validation failed; final model training skipped and predictions remain frozen V5.
V7A final accepted: False | changed rows: 0


## Export reproducible audit bundle
The export always runs. If any gate failed, `submission_v7a.csv` is exactly frozen V5 while all diagnostics document why V7A abstained.

In [11]:
submission=sample[["id"]].copy(); submission["label"]=final_v7a.astype(int)
assert len(submission)==EXPECTED_TEST_ROWS and submission.id.astype(str).tolist()==test.id.astype(str).tolist() and set(submission.label.unique()).issubset({0,1})

train_diag=pd.DataFrame({
    "id":train.id,"label":y,"partition":np.where(np.isin(np.arange(len(train)),lock_idx),"LOCK","DESIGN"),"design_fold":design_fold,
    "v5_label":oof_v5,"v4_rule":oof_v4_rule,"v5_rescue":oof_v5_rescue.astype(int),"candidate":train_candidate.astype(int),
    "mean_nll":design_mean,"top2_nll":design_top2,"risk":design_risk,"v7a_design_label":np.where(np.isin(np.arange(len(train)),design_idx),design_pred,oof_v5)
})

test_scores=pd.DataFrame({
    "id":test.id,"v5_label":test_v5_label,"v4_rule":test_v4_rule,"v5_rescue":test_v5_rescue.astype(int),"candidate":test_candidate.astype(int),
    "mean_nll":test_mean,"top2_nll":test_top2,"risk":test_risk,"veto":proposed.astype(int),"label":final_v7a.astype(int)
})
changed=test_scores.loc[test_scores.veto==1].merge(test[["id","title","content"]],on="id",how="left")
all_history=pd.concat([training_history,pd.DataFrame(final_seed_history)],ignore_index=True) if final_seed_history else training_history.copy()

config={
    "version":"v7a-indobart-conditional-headline-likelihood-verifier","seed":SEED,"lock_seed":LOCK_SEED,"base":"frozen-v5",
    "model":MODEL_NAME,"revision":MODEL_REVISION,"model_source":model_source,"source_max":SOURCE_MAX,"target_max":TARGET_MAX,
    "training":{"epochs":EPOCHS,"lr":LR,"weight_decay":WEIGHT_DECAY,"warmup_ratio":WARMUP_RATIO,"train_batch":TRAIN_BATCH,"grad_accum":GRAD_ACCUM,"effective_batch":TRAIN_BATCH*GRAD_ACCUM,"pair_lambda":PAIR_LAMBDA,"gradient_checkpointing":USE_GRADIENT_CHECKPOINTING,"synthetic":False,"external_data":False},
    "risk":{"mean_weight":RISK_MEAN_WEIGHT,"top2_weight":RISK_TOP2_WEIGHT,"coverage_grid":COVERAGE_GRID.tolist(),"selected_coverage":SELECTED_COVERAGE},
    "gates":{"design_min_gain":MIN_DESIGN_GAIN,"min_positive_design_folds":MIN_POSITIVE_DESIGN_FOLDS,"min_worst_fold_gain":MIN_WORST_DESIGN_FOLD_GAIN,"min_pair_rank_acc":MIN_PAIR_RANK_ACC,"min_lock_precision":MIN_LOCK_PRECISION,"min_bootstrap_positive":MIN_BOOTSTRAP_POSITIVE,"test_changes":[MIN_TEST_CHANGES,MAX_TEST_CHANGES]},
    "status":{"design_ok":DESIGN_OK,"lock_opened":LOCK_OPENED,"lock_ok":LOCK_OK,"validated":V7A_VALIDATED,"test_count_ok":TEST_COUNT_OK,"accepted":V7A_ACCEPTED}
}
provenance={
    "train_path":str(train_path),"test_path":str(test_path),"train_id_sha256":TRAIN_ID_SHA256,"test_id_sha256":TEST_ID_SHA256,
    "frozen_blob_sha256":"e1480451c5d635d180201ae1957c75f6ec0383c99be9f5fb930928c824c29073",
    "torch":torch.__version__,"transformers":transformers.__version__,"sklearn":sklearn.__version__,"gpu_count":GPU_COUNT,"gpus":[torch.cuda.get_device_name(i) for i in range(GPU_COUNT)],
    "timestamp_utc":pd.Timestamp.utcnow().isoformat()
}

artifacts={
    "submission_v7a.csv":submission,
    "design_split_v7a.csv":split_table,
    "design_oof_scores_v7a.csv":train_diag,
    "design_coverage_metrics_v7a.csv":coverage_metrics,
    "design_fold_metrics_v7a.csv":design_fold_metrics,
    "pair_ranking_metrics_v7a.csv":pair_metrics,
    "lock_metrics_v7a.csv":lock_metrics,
    "lock_scores_v7a.csv":lock_scores,
    "lock_bootstrap_v7a.csv":bootstrap_df,
    "training_history_v7a.csv":all_history,
    "test_scores_v7a.csv":test_scores,
    "changed_rows_v7a.csv":changed,
}
for name,df in artifacts.items(): df.to_csv(OUTPUT_DIR/name,index=False)
(OUTPUT_DIR/"config_v7a.json").write_text(json.dumps(config,indent=2,default=str))
(OUTPUT_DIR/"provenance_v7a.json").write_text(json.dumps(provenance,indent=2,default=str))

manifest={}
for p in sorted(OUTPUT_DIR.iterdir()):
    if p.is_file() and not p.name.endswith(".pt"):
        manifest[p.name]={"bytes":p.stat().st_size,"sha256":hashlib.sha256(p.read_bytes()).hexdigest()}
(OUTPUT_DIR/"manifest_v7a.json").write_text(json.dumps(manifest,indent=2))
zip_path=OUTPUT_DIR/"results_v7a.zip"
with zipfile.ZipFile(zip_path,"w",compression=zipfile.ZIP_DEFLATED) as z:
    for p in sorted(OUTPUT_DIR.iterdir()):
        if p.is_file() and p.name!="results_v7a.zip" and not p.name.endswith(".pt"): z.write(p,arcname=p.name)

# Temporary fold checkpoints are not exported; all training/config provenance is preserved.
shutil.rmtree(TMP_DIR,ignore_errors=True)
print("Submission:",OUTPUT_DIR/"submission_v7a.csv")
print("Results bundle:",zip_path)
print("Final label distribution:",submission.label.value_counts().sort_index().to_dict(),"| changed vs V5:",int((submission.label.to_numpy()!=test_v5_label).sum()))

Submission: /kaggle/working/ifest_v7a_output/submission_v7a.csv
Results bundle: /kaggle/working/ifest_v7a_output/results_v7a.zip
Final label distribution: {0: 379, 1: 3224} | changed vs V5: 0
